![Week 5 Day 4](../assets/banners/w5d4.png)

# Week 5 Day 4 - 🧱 The Orchestrator: GradeWise End to End, plus LangSmith

Look at what you have now. A grading agent with tools and guardrails (Days 1–2). Report cards, storage and a results agent (Day 3). A transcription subgraph that reads handwriting (week 4). An exam pack approved by Emma (week 3).

Yesterday we glued some of it together with a `for` loop. That's fine in a notebook, and hopeless in production: what happens when student 7's PDF fails half-way? When a flagged answer needs Emma, and she's gone home? When you want to know *why* a batch of 30 scripts took twenty minutes and cost four dollars?

Today you meet the piece that answers all three: **the orchestrator**, one LangGraph graph from a pile of answer sheets to stored report cards, with checkpoints, a human in the loop, and progress events. Then we switch on **LangSmith** and look inside a full run.

![Week 5 roadmap](../assets/roadmap/week5.png)

### 🎯 By the end of this lab you will
1. 🗺️ Read `gradewise/orchestrator.py`: its stages, its state and why it's shaped that way
2. 📡 Follow a batch live through the `on_event(stage, student_id, detail)` contract
3. ⏸ Pause at `human_review`, inspect the interrupt, "restart Python", and **resume with a teacher override**
4. 🔭 Set up LangSmith tracing, and know what to look for when you debug a full run and its cost

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | 🧱 Day 4 - The Orchestrator: GradeWise End to End | 🧪 |
| 2 | Day 4 - LangSmith: Tracing, Costs and Debugging a Full Run | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">The rehearsal runs (Parts 3–5) use the 🎭 <b>stand-in grader</b> and already-transcribed answer sheets, so the <i>whole</i> orchestrator runs for free: fan-out, interrupt, resume, report cards and storage. Everything goes into a sandbox, <code>output/week5/day4/</code>. The 🔑 cells use your real models (Part 6 reads a real handwritten PDF), and LangSmith cells only do something if you've added a LangSmith key.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import display

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, settings
from gradewise.console import success
from gradewise.evals.compat import import_pandas
from gradewise.ingestion import ingest
from gradewise.rag import get_retriever
from gradewise.schemas import ExamPaper, ReportCard, TranscribedAnswer

pd = import_pandas()  # 💡 = `import pandas as pd`, with a fix for locked-down Windows machines

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
FULL_EXAM = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTIONS = {q.id: q for q in FULL_EXAM.questions}  # every question, for the one-answer demos
TRUTH = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))

# 📦 What the batch runs grade: all 15 parts and 10 students, or Q1–Q3 and 2 students with DATA_SIZE=small
exam = datasets.subset_exam(FULL_EXAM)
STUDENTS = datasets.synthetic_students()

# 🆓 A keyword-only (BM25) copy of the CS201 notes: fake embeddings, real keyword search, zero cost
OFFLINE_DB = OUTPUT_DIR / "week5_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
ingest(
    collection="cs201_offline",
    embeddings=fake_embeddings,
    persist_dir=OFFLINE_DB,
    quiet=True,
    files=datasets.kb_files(),
)  # 📦 all 4 PDFs, or unit1 + the handbook with DATA_SIZE=small
bm25 = get_retriever("cs201_offline", k=4, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

success(
    "Ready",
    {
        "Exam": f"{exam.title} ({len(exam.questions)} questions, {exam.total_marks:g} marks)",
        "Students": f"{len(STUDENTS)} synthetic scripts",
        "Offline KB": f"cs201_offline (BM25, {len(datasets.kb_files())} PDFs)",
        "Grading model": settings.grading_model,
    },
)

In [ ]:
import html

from IPython.display import HTML

from gradewise.reports import render_html


def show_report(report: ReportCard, height: int = 760, paper: ExamPaper | None = None) -> None:
    """Render a report card inline. 💡 Inside an <iframe>, so its CSS can't restyle this notebook."""
    page = render_html(report, paper or exam)
    frame = f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:{height}px;border:0"></iframe>'
    display(HTML(f"<div>{frame}</div>"))

## Part 1: The orchestrator graph

Here's the design from the course plan, the one every week has been building towards:

![Diagram](../assets/diagrams/259fa74da684.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
stateDiagram-v2
  [*] --> load_exam
  load_exam --> transcribe: Send × students
  transcribe --> review_transcription
  review_transcription --> grade: Send × students
  grade --> aggregate
  aggregate --> human_review: flags
  aggregate --> write_report: clear
  human_review --> write_report
  write_report --> store
  store --> [*]
```

</details>


Open `gradewise/orchestrator.py` and find `build_orchestrator()`. Let's build one with the stand-in grader and our sandbox stores, and ask LangGraph to draw it:

In [ ]:
from gradewise.display import show_graph

from gradewise.grading.fakes import KeywordGraderModel
from gradewise.orchestrator import (
    build_orchestrator,
    grade_batch,
    resume_batch,
    sheets_from_transcripts,
    sqlite_checkpointer,
)

SANDBOX = OUTPUT_DIR / "week5" / "day4"
REHEARSAL = {  # 🎭 stand-in grader + free stores: the plumbing is real, the marks are keyword matching
    "llm": KeywordGraderModel(),
    "retriever": bm25,
    "feedback": "rules",
    "db": SANDBOX / "gradewise.db",
    "embeddings": fake_embeddings,
    "persist_dir": SANDBOX / "vector_db",
    "reports_dir": SANDBOX / "reports",
}
app = build_orchestrator(sqlite_checkpointer(SANDBOX / "checkpoints.db"), **REHEARSAL)
show_graph(app)

Five design decisions are worth understanding before we run it:

1. **Two fan-outs.** `load_exam` sends one `transcribe` per student; `review_transcription` sends one `grade` per student. Each `grade` node runs Day 2's grading graph *inside* it (a graph inside a graph), which fans out again per question.
2. **Join nodes.** `review_transcription` and `aggregate` sit after the fan-outs so the next decision sees **every** student. (A conditional edge straight after `grade` would run once per branch, on partial state.)
3. **A reducer keyed by student.** `sheets`, `graded`, `reports` and `files` use `merge_by_student`, a dict merge, not `operator.add`. Re-run a node after a crash and it overwrites its own entry instead of adding a duplicate: every step is **idempotent**.
4. **Plain-dict state.** The state holds `model_dump(mode="json")` dicts, not Pydantic objects, because a checkpointer writes it to disk (and the week 6 API sends it to a browser).
5. **Interrupts, not input().** `review_transcription` and `human_review` call `interrupt(payload)`. With a `SqliteSaver`, a paused batch lives in a file: Emma can close her laptop and resume tomorrow.

In [ ]:
from typing import get_type_hints

from gradewise.orchestrator import OrchestratorState

for key, hint in get_type_hints(OrchestratorState, include_extras=True).items():
    print(f"{key:<8} {hint}".replace("typing.", ""))

## Part 2: The `on_event` contract

Every node reports progress with `emit(stage, student_id, message, **extra)` (Day 2's `gradewise/grading/events.py`), which writes to LangGraph's `custom` stream. `grade_batch()` relays each event to a callback you pass as `on_event`:

```python
def on_event(stage: str, student_id: str | None, detail: dict) -> None: ...
```

- `stage`: one of `load_exam · transcribe · review_transcription · grade · grade_question · aggregate · human_review · write_report · store · done`, plus `paused` and `resumed` from `grade_batch` itself
- `student_id`: who it's about, or `None` for batch-level events
- `detail`: a dict that **always** has `"message"`; other keys depend on the stage (`awarded`, `needs_review`, file paths…)

This tiny contract is all the week 6 UI needs for a live progress view. Ours prints a line per event:

In [ ]:
ICONS = {
    "load_exam": "📝",
    "transcribe": "✍️",
    "review_transcription": "🔎",
    "grade": "🧮",
    "grade_question": "   ·",
    "aggregate": "📊",
    "human_review": "🧑‍🏫",
    "write_report": "📄",
    "store": "💾",
    "paused": "⏸",
    "resumed": "▶️",
    "done": "✅",
}


def on_event(stage: str, student_id: str | None, detail: dict) -> None:
    if stage == "grade_question" and not detail.get("needs_review"):
        return  # 💡 keep the output short: only show flagged questions
    who = f"{student_id} " if student_id else ""
    print(f"{ICONS.get(stage, '▸')} {stage:<20} {who}{detail['message']}")

## Part 3: 🎭 Rehearsal: a batch that needs Emma

Three students from the synthetic set: Sam (student 05), student 08, and student 10, whose Q8 contains the prompt injection from Day 2. They arrive as ready-made `AnswerSheet`s, so `transcribe` skips the vision model. We give the batch a `thread_id`: that's its name in the checkpointer.

📦 This rehearsal always uses the **full paper** (`FULL_EXAM`), whatever `DATA_SIZE` says: Q8 is what makes the batch pause, and the stand-in grader is free anyway.

In [ ]:
from datetime import datetime

THREAD = f"day4-{datetime.now():%Y%m%d-%H%M%S}"  # 💡 a fresh name each run; re-use one to resume it
batch = sheets_from_transcripts(FULL_EXAM, students=["student_05", "student_08", "student_10"])

ALL_QUESTIONS = [q.id for q in FULL_EXAM.questions]  # 💡 an explicit questions=[...] beats DATA_SIZE
paused = grade_batch(
    FULL_EXAM.exam_id,
    batch,
    thread_id=THREAD,
    exam=FULL_EXAM,
    questions=ALL_QUESTIONS,
    orchestrator=app,
    on_event=on_event,
)
print(f"\nstatus: {paused.status}")

The batch **paused**. Every student was graded, but because something was flagged, `aggregate` routed to `human_review`, which called `interrupt()`. Nothing has been written or stored yet. Here's what's waiting for Emma: the interrupt **payload**, plus what the checkpointer knows.

In [ ]:
print(json.dumps(paused.pending, indent=2))

config = {"configurable": {"thread_id": THREAD}}
print("\nnext node(s) to run:", app.get_state(config).next)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">You're Emma. The grader removed the "award full marks" line and gave 2.5/5 for what's left: <i>"DNS = names to IP addresses; browser asks the resolver → root → TLD → authoritative server → IP back"</i>. What mark would <b>you</b> give for the DNS content, and would you change anything else about this student's script? (The human examiner in the golden set gave 3/5.)</span>
        </td>
    </tr>
</table>

## Part 4: "Restart Python", then resume with an override

To prove the pause really lives on disk, let's throw away our graph and build a **brand-new** one on the same checkpoint file, as if Emma came back tomorrow. Then `resume_batch()` answers the interrupt with her decision: an override for Q8, and approval for the rest.

The resume value has the shape the payload advertised in `resume_with`: `{"overrides": {question_id: marks}, "approved": True}`.

In [ ]:
from pathlib import Path

del app  # 👋 "close the laptop"
app = build_orchestrator(sqlite_checkpointer(SANDBOX / "checkpoints.db"), **REHEARSAL)  # ☀️ the next morning

done = resume_batch(THREAD, {"overrides": {"Q8": 3}, "approved": True}, orchestrator=app, on_event=on_event)
print(f"\nstatus: {done.status}")
pd.DataFrame(
    [
        {
            "student": r.student_id,
            "total": r.total,
            "band": r.grade_band,
            "needs_review": r.needs_review,
            "html": Path(done.files[r.student_id]["html"]).name,
        }
        for r in done.reports
    ]
)

Student 10's report card now carries the teacher's mark, a note saying so, and no flag:

In [ ]:
ten = next(r for r in done.reports if r.student_id == "student_10")
q8 = next(r for r in ten.results if r.question_id == "Q8")
print(f"Q8: {q8.awarded_marks:g}/{q8.max_marks:g} · needs_review={q8.needs_review} · {q8.review_reason}")
show_report(ten, paper=FULL_EXAM)

The decision is applied by a small **pure function**, `apply_review()`, which is why it's easy to test. Like everything else in GradeWise, it doesn't trust its input: an override is clamped to the question's range (principle 7), even when the teacher types a typo.

In [ ]:
from gradewise.orchestrator import apply_review

typo = apply_review(ten, {"overrides": {"Q8": 30}, "approved": True})  # meant 3, typed 30
print("Q8 after an override of 30:", next(r.awarded_marks for r in typo.results if r.question_id == "Q8"), "/ 5")

## Part 5: The other pause: low-confidence handwriting

The first interrupt in the graph is `review_transcription`. If the vision model was unsure about an answer (confidence below week 4's threshold of 0.75), the batch pauses **before grading**, so Emma can correct the text instead of a grader marking a misreading. Let's fake an unsure transcription for student 07's Q3, then resume with a correction. (`auto_approve=True` on the resume answers any later pause automatically: handy for demos, never for real marks.)

In [ ]:
unsure = sheets_from_transcripts(exam, students=["student_07"])[0]
unsure = unsure.model_copy(
    update={
        "answers": [
            a.model_copy(update={"confidence": 0.42, "illegible_spans": ["c[?]rcular"]}) if a.question_id == "Q3" else a
            for a in unsure.answers
        ]
    }
)

THREAD_2 = f"{THREAD}-tx"
p2 = grade_batch(exam.exam_id, [unsure], thread_id=THREAD_2, exam=exam, orchestrator=app, on_event=on_event)
print(f"\n⏸ {p2.pending['type']}: {[(i['question_id'], i['confidence'], i['text'][:60]) for i in p2.pending['items']]}")

fixed_text = TRUTH["student_07"]["Q3"]  # Emma reads the scan and types what she sees
d2 = resume_batch(THREAD_2, {"approved": True, "corrections": {"Q3": fixed_text}}, orchestrator=app, auto_approve=True)
print(f"▶️ {d2.status}: {d2.reports[0].student_id} {d2.reports[0].total:g}/{d2.reports[0].max_total:g}")

## Part 6: 🔑 The real thing: handwriting to report card

Now everything real: the **vision model** reads Sam's handwritten PDF (week 4), your **grading model** marks it, and your chat model writes the feedback. With `gpt-6-luna` one student costs a few cents (less with `DATA_SIZE=small`, which grades only Q1–Q3); on `MODEL_MODE=free` it's $0 but slower, because OpenRouter's free models (20 requests a minute, 50 a day: use `DATA_SIZE=small`) are rate-limited (GradeWise retries `429`s with backoff and runs at most two calls in parallel). We keep the free BM25 retriever (pass `retriever=None` to use your week 2 hybrid store instead), and a `reviewer` callback answers any pause by approving, so the cell runs to the end.

💡 `reviewer(payload) → decision` is the non-interactive cousin of a human: return a resume dict, or `None` to stop and leave the batch paused.

In [ ]:
from gradewise.errors import GradeWiseError

SHEET_PDF = DATA_DIR / "answer-sheets" / "synthetic" / "student_05.pdf"
REAL = {
    "retriever": bm25,
    "db": SANDBOX / "real" / "gradewise.db",
    "persist_dir": SANDBOX / "real" / "vector_db",
    "reports_dir": SANDBOX / "real" / "reports",
}


def approve_everything(payload: dict) -> dict:
    print(f"🧑‍🏫 (auto-reviewer) approving {payload['type']} for {payload['student_id']}")
    return {"approved": True, "overrides": {}, "corrections": {}}


try:
    real = grade_batch(
        exam.exam_id,
        [SHEET_PDF],
        thread_id=f"{THREAD}-real",
        exam=exam,
        checkpointer="memory",
        on_event=on_event,
        reviewer=approve_everything,
        **REAL,
    )
    show_report(real.reports[0])
except GradeWiseError as err:
    err.show()

## Part 7: 🔭 LangSmith: tracing, costs and debugging a full run

A batch of three students is already 45 grading agents, each with its own model and tool calls, inside two levels of fan-out. When one mark looks wrong, or the run is slow or expensive, `print()` won't save you. **LangSmith** records every run as a tree you can click through:

![Diagram](../assets/diagrams/360f93bc84b7.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart TD
  B[grade_batch · thread day4-…] --> T[transcribe × 3]
  B --> G[grade × 3]
  G --> GG[grading graph]
  GG --> Q1[grade_question Q1] & Q2[grade_question …] & Q8[grade_question Q8 ⚑]
  Q8 --> M1[🤖 model: retrieve_reference?] --> TL[🧰 retrieve_reference] --> M2[🤖 model: GradeResult]
  B --> H[human_review ⏸] --> W[write_report] --> S[store]
```

</details>


### Setup (two minutes, free tier)
1. Create an account at [smith.langchain.com](https://smith.langchain.com/) and make an API key (*Settings → API Keys*).
2. In your `.env` (the template is already in `.env.example`):
   ```bash
   LANGSMITH_TRACING=true
   LANGSMITH_API_KEY=lsv2_...your key...
   LANGSMITH_PROJECT=gradewise
   ```
3. **Save the file and restart the kernel.** That's it: LangChain and LangGraph read these variables and trace every run automatically, no code changes.

⚠️ Traces contain everything the models saw: here, **students' answers**. For real scripts, check your school's data-protection rules before sending them to any third-party service (LangSmith can also be self-hosted).

In [ ]:
from gradewise.config import get_settings

get_settings.cache_clear()  # 💡 settings are cached: clear them to re-read .env, in case you just edited it
ls = get_settings()
TRACING = bool(ls.langsmith_api_key) and ls.langsmith_tracing
print(f"LANGSMITH_TRACING={ls.langsmith_tracing} · key set: {bool(ls.langsmith_api_key)}")
print(f"LANGSMITH_PROJECT={ls.langsmith_project}")
if not TRACING:
    print("ℹ️  Tracing is off: the next cell will skip. Add the three lines above to .env and restart the kernel.")

The next cell runs a small rehearsal batch (stand-in grader: **free** apart from LangSmith's free tier) inside `tracing_context()`, which adds **tags** and **metadata** to every run in the tree, so you can filter for it later ("show me every week5 rehearsal of thread X"). It only runs if your key is set.

In [ ]:
if TRACING:
    from langsmith import tracing_context

    traced_thread = f"{THREAD}-traced"
    with tracing_context(
        project_name=ls.langsmith_project,
        tags=["week5", "day4", "rehearsal"],
        metadata={"thread_id": traced_thread, "students": 2},
    ):
        traced = grade_batch(
            exam.exam_id,
            sheets_from_transcripts(exam, students=2),
            thread_id=traced_thread,
            exam=exam,
            orchestrator=app,
            auto_approve=True,
        )
    print(
        f"✅ {traced.status}. Open https://smith.langchain.com → project '{ls.langsmith_project}' → filter tag 'day4'"
    )
else:
    print("⏭  skipped (no LangSmith key)")

### Debugging a full run: what to look at
| Question | Where in the trace |
|---|---|
| **Why did Q8 get this mark?** | Open that `grade_question` branch → the model's messages: which tools did it call, what came back, what `GradeResult` did it return *before* the guardrails? |
| **Why was it slow?** | Sort runs by latency: is it one slow student (a 6-page PDF), or every `grade_question` waiting on rate limits? (Tune `MAX_PARALLEL_QUESTIONS` / `MAX_PARALLEL_STUDENTS`.) |
| **What did it cost?** | Each model run shows its tokens, and the cost for models LangSmith knows the price of; the root run adds them up. Which stage dominates: vision, grading or feedback? |
| **Did the grader loop?** | Count the model calls inside one agent: more than a handful means a prompt problem (and `ModelCallLimitMiddleware` cut it off at 8). |
| **Did a change make things worse?** | Compare two traces of the same student before and after a prompt edit. (On Day 5 we'll measure it properly.) |

### Counting tokens without LangSmith
You can also count tokens in-process. `get_usage_metadata_callback()` collects `usage_metadata` from every model call made inside its `with` block, per model. 🔑 Grade one answer for real and multiply up. The prices are `gpt-6-luna`'s, the `MODEL_MODE=cheap` default ($0.10 in / $0.50 out per million tokens); on `MODEL_MODE=free` the bill is $0, and if you switch to `MODEL_MODE=custom` with `SMART_MODEL=openrouter:openai/gpt-6.1-terra`, use $1 / $5.

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

from gradewise.grading import grade_answer

try:
    q3 = QUESTIONS["Q3"]
    with get_usage_metadata_callback() as usage:
        grade_answer(
            q3,
            TranscribedAnswer(question_id="Q3", text=TRUTH["student_05"]["Q3"], confidence=1.0),
            retriever=bm25,
            exam=exam,
        )
    PRICE_IN, PRICE_OUT = 0.10, 0.50  # 💰 US$ per 1M tokens: gpt-6-luna (edit for your model)
    answers = len(FULL_EXAM.questions) * 30
    for model, u in usage.usage_metadata.items():
        print(f"{model}: {u['input_tokens']:,} in + {u['output_tokens']:,} out for ONE answer")
        dollars = (u["input_tokens"] * PRICE_IN + u["output_tokens"] * PRICE_OUT) / 1e6 * answers
        print(
            f"≈ ${dollars:.2f} to grade a class of 30 ({answers} answers) at ${PRICE_IN} / ${PRICE_OUT} per 1M tokens"
        )
except GradeWiseError as err:
    err.show()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">The orchestrator is what turns a clever demo into a <b>product</b>. Checkpoints mean a crash at script 27 of 30 costs you one script, not thirty. Interrupts mean a teacher's judgement is built into the workflow, not bolted on. Progress events mean a UI. And tracing means that when a parent questions a mark, you can show exactly what the grader saw and did, months later. That audit trail is often what a school's procurement team asks about first.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">This is the M5 milestone, <code>gradewise/orchestrator.py</code>: <code>build_orchestrator()</code>, <code>grade_batch()</code>, <code>resume_batch()</code>, <code>sheets_from_transcripts()</code>, <code>apply_review()</code>. From the terminal (it asks <i>you</i> at every pause):<br/><br/><code>uv run week5/project/orchestrator.py --fake --students 3</code> (free, writes to <code>output/fake_run/</code>)<br/><code>uv run week5/project/orchestrator.py --from-transcripts --students 3</code> (real grader, no vision)<br/><code>uv run week5/project/orchestrator.py --students 2</code> (everything real, from the handwritten PDFs)<br/><br/>Tests: <code>uv run python -m pytest tests/test_orchestrator.py -q</code>.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🗺️ read the orchestrator: two fan-outs, join nodes, an idempotent per-student reducer, plain-dict state and two interrupts
- 📡 followed a batch live through the `on_event(stage, student_id, detail)` contract
- ⏸ paused at `human_review`, "restarted Python", and resumed with Emma's override, which `apply_review()` clamps
- 🔎 corrected a low-confidence transcription before it was graded
- 🔭 set up LangSmith, traced a run with tags and metadata, and counted tokens in-process

**Tomorrow (Day 5):** the question that matters most: **is GradeWise any good?** We compare its marks with a human examiner's on a golden set (MAE, exact agreement and quadratic weighted kappa, worked out by hand first), measure transcription with CER and WER, and lock the results in with regression tests. Then we wrap up the week.

⏭ **Next:** `week5/day5.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>A smarter reviewer.</b> <code>auto_approve=True</code> is too trusting and a human for every flag is too slow. Write <code>triage_reviewer(payload)</code> for <code>grade_batch(..., reviewer=...)</code>: for <code>human_review</code>, approve automatically when <i>every</i> flagged item's reason is only "low grader confidence", but return <code>None</code> (stay paused for Emma) if any reason mentions prompt injection or illegible text; for <code>review_transcription</code>, approve with no corrections if every confidence is at least 0.6, otherwise return <code>None</code>. Log each decision with a reason. Run it on all ten students with the stand-in and count how many batches still need Emma. Solution: <code>week5/solutions/day4_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangGraph interrupts and human-in-the-loop: <a href="https://docs.langchain.com/oss/python/langgraph/interrupts">docs.langchain.com/oss/python/langgraph/interrupts</a><br/>• LangGraph persistence and checkpointers: <a href="https://docs.langchain.com/oss/python/langgraph/persistence">docs.langchain.com/oss/python/langgraph/persistence</a><br/>• LangSmith tracing quickstart: <a href="https://docs.smith.langchain.com/">docs.smith.langchain.com</a></span>
        </td>
    </tr>
</table>